In [ ]:
from pathlib import Path
import os, json, math, random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from facenet_pytorch import InceptionResnetV1, fixed_image_standardization

In [ ]:
# ===== Paths =====
DATA_DIR   = Path("/home/danila/networks/data")
FRAMES_DIR = DATA_DIR / "frames-1"
TRAIN_CSV  = DATA_DIR / "train_split.csv"
VALID_CSV  = DATA_DIR / "valid_split.csv"

# npz с bbox/face_found/face_prob/frame_numbers/timestamps_sec
FACE_NPZ_DIR = DATA_DIR / "embeddings" / "faces_emotiefflib_fps5_v1"
assert FACE_NPZ_DIR.exists(), f"FACE_NPZ_DIR not found: {FACE_NPZ_DIR}"

RUN_DIR = DATA_DIR / "runs" / "video_trainable_encoder_faces_v1"
RUN_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH = RUN_DIR / "best_by_corr.pt"
HIST_PATH = RUN_DIR / "history.json"
MISS_PATH = RUN_DIR / "missing_or_bad.csv"

# ===== Task =====
EMOTIONS = ["Admiration","Amusement","Determination","Empathic Pain","Excitement","Joy"]
NUM_TARGETS = len(EMOTIONS)

# ===== IDs / FPS / Crop window =====
ID_WIDTH = 5
TARGET_FPS = 5

# >>> ВАЖНО: можно менять длительность окна тут
MAX_SEC = 20.0
# MAX_SEC = 12.0

# max frames in window (для 5 fps)
MAX_FRAMES = int(MAX_SEC * TARGET_FPS + 1e-6)

# ===== Face crop =====
FACE_MARGIN = 0.20
IMG_SIZE = 160

# ===== Quality features (для attn bias) =====
FRAME_W = 1280
FRAME_H = 720

# ===== Data filters =====
MIN_VALID_FRAMES = 5

# ===== Training =====
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

BATCH_SIZE = 2
NUM_WORKERS = 4
PIN_MEMORY = True

# encoder (trainable)
ENC_LR = 3e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = 0.0
EPOCHS = 40
PATIENCE = 8
MIN_DELTA = 1e-4

# schedule
USE_COSINE = True
WARMUP_STEPS = 0

# amp
USE_AMP = (DEVICE == "cuda")
GRAD_CLIP = 1.0

# loss
LOSS_MODE = "pearson+mse"      # "mse" / "pearson" / "pearson+mse"
LAMBDA_MSE = 0.1

# pearson stability
ACCUM_STEPS = 4                # effective batch = BATCH_SIZE*ACCUM_STEPS

# backbone freeze option
FREEZE_ENCODER_EPOCHS = 0

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

print("Device:", DEVICE, "| MAX_SEC:", MAX_SEC, "| MAX_FRAMES:", MAX_FRAMES)

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

In [ ]:
def frame_path(video_id: str, frame_number: int) -> Path:
    return FRAMES_DIR / video_id / f"{frame_number:06d}.jpg"

def load_face_npz(path: Path):
    d = np.load(path, allow_pickle=False)
    emb_dummy = d["embeddings"]
    T = emb_dummy.shape[0]

    frame_numbers = d["frame_numbers"].astype(np.int32) if "frame_numbers" in d.files else np.arange(T, dtype=np.int32)
    ts = d["timestamps_sec"].astype(np.float32) if "timestamps_sec" in d.files else None

    bbox = d["bbox_xyxy"].astype(np.float32) if "bbox_xyxy" in d.files else np.full((T,4), -1, np.float32)
    face_prob = d["face_prob"].astype(np.float32) if "face_prob" in d.files else np.ones((T,), np.float32)

    if "face_found" in d.files:
        mask = d["face_found"].astype(bool)
    else:
        # fallback
        mask = np.ones((T,), dtype=bool)

    return frame_numbers, ts, bbox, face_prob, mask

def crop_face_with_margin(img_rgb: np.ndarray, box_xyxy: np.ndarray, margin: float):
    h, w = img_rgb.shape[:2]
    x1, y1, x2, y2 = box_xyxy.astype(np.float32)
    bw = x2 - x1
    bh = y2 - y1
    x1 = x1 - margin * bw
    y1 = y1 - margin * bh
    x2 = x2 + margin * bw
    y2 = y2 + margin * bh
    x1 = int(max(0, math.floor(x1)))
    y1 = int(max(0, math.floor(y1)))
    x2 = int(min(w, math.ceil(x2)))
    y2 = int(min(h, math.ceil(y2)))
    if x2 <= x1 or y2 <= y1:
        return None
    return img_rgb[y1:y2, x1:x2, :]

def quality_from_prob_area(face_prob: np.ndarray, bbox: np.ndarray, mask: np.ndarray):
    x1, y1, x2, y2 = bbox[:,0], bbox[:,1], bbox[:,2], bbox[:,3]
    w = np.clip(x2 - x1, 0, None)
    h = np.clip(y2 - y1, 0, None)
    area = w * h
    area_ratio = area / float(FRAME_W * FRAME_H + 1e-6)
    q = face_prob * np.sqrt(np.clip(area_ratio, 0, 1))
    q = np.clip(q, 0.0, 1.0).astype(np.float32)
    q[~mask] = 0.0
    return q

In [ ]:
train_df = pd.read_csv(TRAIN_CSV, dtype={"Filename": str})
valid_df = pd.read_csv(VALID_CSV, dtype={"Filename": str})

train_df["Filename"] = train_df["Filename"].str.zfill(ID_WIDTH)
valid_df["Filename"] = valid_df["Filename"].str.zfill(ID_WIDTH)

def filter_existing(df: pd.DataFrame):
    keep, miss = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="filter"):
        vid = row["Filename"]
        p = FACE_NPZ_DIR / f"{vid}.npz"
        if not p.exists():
            miss.append({"video_id": vid, "reason": "face_npz_missing"})
            continue
        # frames dir check
        if not (FRAMES_DIR / vid).exists():
            miss.append({"video_id": vid, "reason": "frames_dir_missing"})
            continue
        try:
            frame_numbers, ts, bbox, face_prob, mask = load_face_npz(p)
            # trim to MAX_SEC using timestamps if available
            if ts is not None:
                keep_idx = ts <= float(MAX_SEC) + 1e-6
            else:
                keep_idx = np.arange(len(frame_numbers)) < MAX_FRAMES
            mask2 = mask[keep_idx]
            if int(mask2.sum()) < MIN_VALID_FRAMES:
                miss.append({"video_id": vid, "reason": f"too_few_faces<{MIN_VALID_FRAMES}"})
                continue
        except Exception as e:
            miss.append({"video_id": vid, "reason": f"npz_read_error:{repr(e)}"})
            continue
        keep.append(row)
    return pd.DataFrame(keep).reset_index(drop=True), pd.DataFrame(miss)

train_df2, train_miss = filter_existing(train_df)
valid_df2, valid_miss = filter_existing(valid_df)

print("Train ok:", len(train_df2), "miss:", len(train_miss))
print("Val   ok:", len(valid_df2), "miss:", len(valid_miss))

pd.concat([train_miss.assign(split="train"), valid_miss.assign(split="val")]).to_csv(MISS_PATH, index=False)
print("Saved missing:", MISS_PATH)

train_df, valid_df = train_df2, valid_df2

In [ ]:
y_train = train_df[EMOTIONS].values.astype(np.float32)
y_mean = y_train.mean(axis=0)
y_std  = np.clip(y_train.std(axis=0), 1e-3, None)

np.savez(RUN_DIR / "target_norm.npz", y_mean=y_mean, y_std=y_std, emotions=np.array(EMOTIONS))
print("y_mean:", y_mean)
print("y_std :", y_std)

In [ ]:
class VideoFaceFramesDataset(Dataset):
    def __init__(self, df: pd.DataFrame, training: bool):
        self.df = df.reset_index(drop=True)
        self.training = training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        vid = str(row["Filename"]).zfill(ID_WIDTH)
        npz_path = FACE_NPZ_DIR / f"{vid}.npz"

        frame_numbers, ts, bbox, face_prob, mask = load_face_npz(npz_path)

        # trim to MAX_SEC / MAX_FRAMES
        if ts is not None:
            keep = ts <= float(MAX_SEC) + 1e-6
        else:
            keep = np.arange(len(frame_numbers)) < MAX_FRAMES

        frame_numbers = frame_numbers[keep]
        bbox = bbox[keep]
        face_prob = face_prob[keep]
        mask = mask[keep].astype(bool)

        # hard cap to MAX_FRAMES
        if len(frame_numbers) > MAX_FRAMES:
            frame_numbers = frame_numbers[:MAX_FRAMES]
            bbox = bbox[:MAX_FRAMES]
            face_prob = face_prob[:MAX_FRAMES]
            mask = mask[:MAX_FRAMES]

        T_total = len(frame_numbers)
        q = quality_from_prob_area(face_prob, bbox, mask)

        # targets (normalized)
        y = np.array([row[e] for e in EMOTIONS], dtype=np.float32)
        y = (y - y_mean) / y_std

        # load only valid-face frames, keep their time indices
        frames = []
        time_idx = []
        for t in range(T_total):
            if not mask[t]:
                continue
            fn = int(frame_numbers[t])
            fp = frame_path(vid, fn)
            if not fp.exists():
                continue

            bgr = cv2.imread(str(fp), cv2.IMREAD_COLOR)
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

            crop = crop_face_with_margin(rgb, bbox[t], FACE_MARGIN)
            if crop is None:
                continue

            crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)

            # optional light aug
            if self.training:
                if random.random() < 0.5:
                    crop = crop[:, ::-1, :]  # hflip

            # to tensor [3,H,W] in 0..255
            ten = torch.from_numpy(crop).permute(2,0,1).float()  # 0..255
            ten = fixed_image_standardization(ten)  # (x-127.5)/128
            frames.append(ten)
            time_idx.append(t)

        if len(frames) == 0:
            frames_t = torch.zeros((0, 3, IMG_SIZE, IMG_SIZE), dtype=torch.float32)
            time_idx_t = torch.zeros((0,), dtype=torch.long)
        else:
            frames_t = torch.stack(frames, dim=0)
            time_idx_t = torch.tensor(time_idx, dtype=torch.long)

        return {
            "video_id": vid,
            "frames": frames_t,             # [N_valid,3,H,W]
            "time_idx": time_idx_t,         # [N_valid]
            "T_total": int(T_total),        # scalar
            "mask": torch.tensor(mask, dtype=torch.bool),   # [T_total]
            "q": torch.tensor(q, dtype=torch.float32),      # [T_total]
            "y": torch.tensor(y, dtype=torch.float32),      # [6]
        }

In [ ]:
def collate_packed(batch):
    B = len(batch)
    maxT = max(b["T_total"] for b in batch)

    # pack frames
    frames_all = []
    batch_idx_all = []
    time_idx_all = []

    mask = torch.zeros((B, maxT), dtype=torch.bool)
    q = torch.zeros((B, maxT), dtype=torch.float32)
    y = torch.zeros((B, NUM_TARGETS), dtype=torch.float32)
    vids = []

    for i, b in enumerate(batch):
        vids.append(b["video_id"])
        T = b["T_total"]

        mask[i, :T] = b["mask"]
        q[i, :T] = b["q"]
        y[i] = b["y"]

        if b["frames"].numel() > 0:
            n = b["frames"].shape[0]
            frames_all.append(b["frames"])
            batch_idx_all.append(torch.full((n,), i, dtype=torch.long))
            time_idx_all.append(b["time_idx"])

    if frames_all:
        frames_all = torch.cat(frames_all, dim=0)         # [N,3,H,W]
        batch_idx_all = torch.cat(batch_idx_all, dim=0)   # [N]
        time_idx_all = torch.cat(time_idx_all, dim=0)     # [N]
    else:
        frames_all = torch.zeros((0,3,IMG_SIZE,IMG_SIZE), dtype=torch.float32)
        batch_idx_all = torch.zeros((0,), dtype=torch.long)
        time_idx_all = torch.zeros((0,), dtype=torch.long)

    return {
        "video_id": vids,
        "frames": frames_all,
        "batch_idx": batch_idx_all,
        "time_idx": time_idx_all,
        "mask": mask,
        "q": q,
        "y": y,
    }

train_ds = VideoFaceFramesDataset(train_df, training=True)
valid_ds = VideoFaceFramesDataset(valid_df, training=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                          collate_fn=collate_packed, drop_last=False)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                          collate_fn=collate_packed, drop_last=False)

b = next(iter(train_loader))
print("Packed frames:", b["frames"].shape, "mask:", b["mask"].shape)

In [ ]:
class TCNBlock(nn.Module):
    def __init__(self, d: int, kernel: int, dilation: int, dropout: float):
        super().__init__()
        padding = (kernel - 1) * dilation // 2
        self.conv1 = nn.Conv1d(d, d, kernel_size=kernel, dilation=dilation, padding=padding)
        self.conv2 = nn.Conv1d(d, d, kernel_size=kernel, dilation=dilation, padding=padding)
        self.norm1 = nn.GroupNorm(1, d)
        self.norm2 = nn.GroupNorm(1, d)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        res = x
        x = self.drop(F.gelu(self.norm1(self.conv1(x))))
        x = self.drop(F.gelu(self.norm2(self.conv2(x))))
        return x + res

class TCNEncoder(nn.Module):
    def __init__(self, d: int, layers: int, kernel: int, dropout: float):
        super().__init__()
        self.blocks = nn.ModuleList([TCNBlock(d, kernel, 2**i, dropout) for i in range(layers)])

    def forward(self, x):
        x = x.transpose(1,2)  # [B,d,T]
        for b in self.blocks:
            x = b(x)
        return x.transpose(1,2)  # [B,T,d]

class QualityAttentiveStatsPooling(nn.Module):
    def __init__(self, d: int, attn_hidden: int, dropout: float, temp: float = 1.5):
        super().__init__()
        self.temp = temp
        self.attn = nn.Sequential(
            nn.Linear(d, attn_hidden),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(attn_hidden, 1),
        )
        self.q_alpha = nn.Parameter(torch.tensor(0.5))

    def forward(self, x, mask, q):
        logits = self.attn(x).squeeze(-1) / self.temp
        qb = torch.log(q.clamp_min(1e-4))
        logits = logits + self.q_alpha * qb
        logits = logits.masked_fill(~mask, -1e4)

        w = torch.softmax(logits, dim=1)
        w = w * mask.float()
        w = w / (w.sum(dim=1, keepdim=True) + 1e-6)
        w = w.unsqueeze(-1)

        mu = (w * x).sum(dim=1)
        var = (w * (x - mu.unsqueeze(1)).pow(2)).sum(dim=1)
        std = torch.sqrt(var + 1e-6)
        return torch.cat([mu, std], dim=-1)

class VideoTrainableFaceModel(nn.Module):
    def __init__(self, d_model=256, tcn_layers=4, tcn_kernel=3, dropout=0.2, attn_hidden=128, attn_temp=1.5):
        super().__init__()
        # Trainable face encoder
        self.face_enc = InceptionResnetV1(pretrained="vggface2", classify=False)
        enc_dim = 512  # output dim of InceptionResnetV1 embeddings

        self.proj = nn.Sequential(
            nn.Linear(enc_dim, d_model),
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
        )
        self.tcn = TCNEncoder(d_model, tcn_layers, tcn_kernel, dropout)
        self.pool = QualityAttentiveStatsPooling(d_model, attn_hidden, dropout, temp=attn_temp)
        self.head = nn.Sequential(
            nn.Linear(2*d_model, 2*d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(2*d_model, NUM_TARGETS),
        )

    def forward(self, frames, batch_idx, time_idx, mask, q):
        """
        frames: [N,3,H,W]
        batch_idx/time_idx: [N]
        mask/q: [B,T]
        """
        B, T = mask.shape

        # encode only valid frames
        feats = self.face_enc(frames)  # [N,512]

        # scatter into [B,T,512]
        x = torch.zeros((B, T, feats.shape[-1]), device=feats.device, dtype=feats.dtype)
        x[batch_idx, time_idx] = feats

        x = self.proj(x)
        x = self.tcn(x)
        z = self.pool(x, mask, q)
        out = self.head(z)
        return out

model = VideoTrainableFaceModel(
    d_model=256,
    tcn_layers=4,
    tcn_kernel=3,
    dropout=0.3,
    attn_hidden=128,
    attn_temp=1.5
).to(DEVICE)

print(model)

In [ ]:
def pearson_corr_torch(preds: torch.Tensor, targets: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    vx = preds - preds.mean(0)
    vy = targets - targets.mean(0)
    corr = (vx * vy).sum(0) / (torch.sqrt((vx**2).sum(0) * (vy**2).sum(0)) + eps)
    return corr.mean()

mse_fn = nn.MSELoss()

def loss_fn(pred, y):
    if LOSS_MODE == "mse":
        return mse_fn(pred, y)
    if LOSS_MODE == "pearson":
        return 1.0 - pearson_corr_torch(pred, y)
    if LOSS_MODE == "pearson+mse":
        return (1.0 - pearson_corr_torch(pred, y)) + (LAMBDA_MSE * mse_fn(pred, y))
    raise ValueError("Unknown LOSS_MODE")

In [ ]:
# param groups: encoder vs rest
enc_params = list(model.face_enc.parameters())
rest_params = [p for n,p in model.named_parameters() if not n.startswith("face_enc.")]

optimizer = torch.optim.AdamW(
    [
        {"params": enc_params, "lr": ENC_LR},
        {"params": rest_params, "lr": HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY
)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

total_steps = EPOCHS * math.ceil(len(train_loader) / ACCUM_STEPS)
print("Total optimizer steps:", total_steps)

if USE_COSINE:
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_steps))
else:
    scheduler = None

In [ ]:
def to_dev(batch):
    frames = batch["frames"].to(DEVICE, non_blocking=True)
    batch_idx = batch["batch_idx"].to(DEVICE, non_blocking=True)
    time_idx = batch["time_idx"].to(DEVICE, non_blocking=True)
    mask = batch["mask"].to(DEVICE, non_blocking=True)
    q = batch["q"].to(DEVICE, non_blocking=True)
    y = batch["y"].to(DEVICE, non_blocking=True)
    return frames, batch_idx, time_idx, mask, q, y

@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    Ps, Ys = [], []
    for batch in tqdm(loader, desc="eval", leave=False):
        frames, batch_idx, time_idx, mask, q, y = to_dev(batch)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred = model(frames, batch_idx, time_idx, mask, q)
        Ps.append(pred.detach().float().cpu())
        Ys.append(y.detach().float().cpu())

    P = torch.cat(Ps, dim=0)
    Y = torch.cat(Ys, dim=0)
    corr = pearson_corr_torch(P, Y).item()
    loss = loss_fn(P, Y).item()
    return {"corr": float(corr), "loss": float(loss)}

def set_encoder_trainable(model, trainable: bool):
    for p in model.face_enc.parameters():
        p.requires_grad = trainable

def train_epoch(model, loader, epoch: int):
    model.train()

    # optional freeze stage
    if epoch <= FREEZE_ENCODER_EPOCHS:
        set_encoder_trainable(model, False)
    else:
        set_encoder_trainable(model, True)

    buf_pred, buf_y = [], []
    buf_count = 0

    all_pred, all_y = [], []
    losses = []
    opt_steps = 0

    for step, batch in enumerate(tqdm(loader, desc="train", leave=False), start=1):
        frames, batch_idx, time_idx, mask, q, y = to_dev(batch)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred = model(frames, batch_idx, time_idx, mask, q)

        all_pred.append(pred.detach().float().cpu())
        all_y.append(y.detach().float().cpu())

        buf_pred.append(pred)
        buf_y.append(y)
        buf_count += 1

        flush = (buf_count >= ACCUM_STEPS) or (step == len(loader))
        if not flush:
            continue

        optimizer.zero_grad(set_to_none=True)
        P = torch.cat(buf_pred, dim=0)
        Y = torch.cat(buf_y, dim=0)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            loss = loss_fn(P, Y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        if scheduler is not None:
            scheduler.step()

        losses.append(float(loss.item()))
        opt_steps += 1

        buf_pred, buf_y, buf_count = [], [], 0

    P_all = torch.cat(all_pred, dim=0)
    Y_all = torch.cat(all_y, dim=0)
    train_corr = pearson_corr_torch(P_all, Y_all).item()
    train_loss = float(np.mean(losses)) if losses else float("nan")

    lr_enc = optimizer.param_groups[0]["lr"]
    lr_head = optimizer.param_groups[1]["lr"]

    return {"corr": float(train_corr), "loss": float(train_loss), "lr_enc": float(lr_enc), "lr_head": float(lr_head), "opt_steps": opt_steps}

In [ ]:
history = []
best_corr = -1e9
best_epoch = -1
bad = 0

for epoch in range(1, EPOCHS + 1):
    tr = train_epoch(model, train_loader, epoch)
    va = eval_epoch(model, valid_loader)

    row = {"epoch": epoch, "train": tr, "val": va}
    history.append(row)

    print(
        f"Epoch {epoch:03d} | "
        f"train corr {tr['corr']:.4f} loss {tr['loss']:.4f} | "
        f"val corr {va['corr']:.4f} loss {va['loss']:.4f} | "
        f"lr_enc {tr['lr_enc']:.2e} lr_head {tr['lr_head']:.2e}"
    )

    with open(HIST_PATH, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

    improved = (va["corr"] - best_corr) > MIN_DELTA
    if improved:
        best_corr = va["corr"]
        best_epoch = epoch
        bad = 0
        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "best_corr": best_corr,
                "config": {
                    "MAX_SEC": MAX_SEC,
                    "MAX_FRAMES": MAX_FRAMES,
                    "ENC_LR": ENC_LR,
                    "HEAD_LR": HEAD_LR,
                    "LOSS_MODE": LOSS_MODE,
                    "ACCUM_STEPS": ACCUM_STEPS,
                }
            },
            CKPT_PATH
        )
        print(f"  ✅ Saved best checkpoint: epoch={epoch}, val_corr={best_corr:.4f}")
    else:
        bad += 1
        print(f"  ⏳ No improvement: {bad}/{PATIENCE}")

    if bad >= PATIENCE:
        print(f"🛑 Early stopping. Best epoch={best_epoch}, best val corr={best_corr:.4f}")
        break

print("Done. Best:", best_corr, "at epoch", best_epoch)
print("Saved:", CKPT_PATH)

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(ckpt["model_state"])
model.to(DEVICE)

va = eval_epoch(model, valid_loader)
print("Best epoch:", ckpt["epoch"])
print("Val corr:", va["corr"], "| Val loss:", va["loss"])